In [ ]:
!pip install -U transformers datasets accelerate sentencepiece evaluate --quiet

# imports

In [ ]:
import os
import gc
import random
import shutil
import time
from dataclasses import dataclass
from typing import Dict, List, Optional
import evaluate
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader

from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR
from torch.optim.optimizer import Optimizer
import math

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorForSeq2Seq,
)

try:
    from safetensors.torch import load_file as safe_load, save_file as safe_save
except Exception:
    safe_load, safe_save = None, None

from transformers.utils import logging
logging.set_verbosity_error()   # hides WARNING/INFO messages (incl. load reports)


In [ ]:
# from huggingface_hub import login
# login(os.environ["HF_TOKEN"])  # optional; public models/datasets need no token

In [ ]:
def cosine_lr(initial_lr: float,
              round_idx: int,
              total_rounds: int,
              eta_min: float = 0.0) -> float:
    round_idx = min(round_idx, total_rounds)
    cosine_decay = 0.5 * (1 + math.cos(math.pi * round_idx / total_rounds))
    return eta_min + (initial_lr - eta_min) * cosine_decay

# Constants

In [ ]:
seed = 8541865

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)



MODEL_NAME = "google/electra-small-discriminator"
Main_path = "./main_model"


OUTPUT_DIR = "electra"
MAIN_PATH_MODEL = Main_path
global_adapter_dir = Main_path
ADAPTER_BIN = "adapter_model.bin"
ADAPTER_SAFE = "adapter_model.safetensors"

EPOCHS = 1
LR = 2.3e-4
rounds = 100
NUM_LABELS = 2
MAX_LENGTH = 96
BATCH_SIZE = 512
num_of_client = 100
num_of_Part = 10

# sophia

# model Import

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=NUM_LABELS)
model.to(device)
model.save_pretrained(MAIN_PATH_MODEL)
model.save_pretrained("./pre_model")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(trainable_params/total_params)

# Data

In [ ]:
import pandas as pd
test_url='https://huggingface.co/datasets/stanfordnlp/sentiment140/resolve/refs%2Fconvert%2Fparquet/sentiment140/test/0000.parquet'
!wget {test_url}

train_url='https://huggingface.co/datasets/stanfordnlp/sentiment140/resolve/refs%2Fconvert%2Fparquet/sentiment140/train/0000.parquet'
!wget {train_url}

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

In [ ]:
df_test = pd.read_parquet('0000.parquet')
df_train = pd.read_parquet('0000.parquet.1')
df_train.info()
df_test  = df_test[df_test["sentiment"] != 2]
df_train["label"] = (df_train["sentiment"] == 4).astype("int8")
df_test["label"]  = (df_test["sentiment"] == 4).astype("int8")

X = df_train.drop(columns=["label"])
y = df_train["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.005,
    stratify=y,
    random_state=seed
)

train_df = X_train.copy()
train_df["label"] = y_train

test_df = X_test.copy()
test_df["label"] = y_test
print(len(train_df),len(test_df))

In [ ]:
label_counts = df_test["label"].astype(int).value_counts().sort_index()

print(f"Number of data points {len(df_test)}")
print("Number of data points per label in test_dataset:")

for label, count in label_counts.items():
    print(f"Label {label}: {count} data points")

In [ ]:
label_counts = df_train["label"].astype(int).value_counts().sort_index()

print(f"Number of data points {len(df_train)}")
print("Number of data points per label in test_dataset:")

for label, count in label_counts.items():
    print(f"Label {label}: {count} data points")

In [ ]:
label_counts = test_df["label"].astype(int).value_counts().sort_index()

print(f"Number of data points {len(test_df)}")
print("Number of data points per label in test_dataset:")

for label, count in label_counts.items():
    print(f"Label {label}: {count} data points")

In [ ]:
hf_valid = Dataset.from_pandas(test_df, preserve_index=False)
hf_test  = Dataset.from_pandas(df_test, preserve_index=False)

def preprocess_fn(examples):
    encodings = tokenizer(
        examples["text"],
        truncation=True,
        padding=False,
        max_length=MAX_LENGTH,
    )
    encodings["labels"] = examples["label"]
    return encodings

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
cols = ["input_ids", "attention_mask", "labels"]

tokenized_valid = hf_valid.map(preprocess_fn, batched=True, remove_columns=hf_valid.column_names)
tokenized_test  = hf_test.map(preprocess_fn, batched=True, remove_columns=hf_test.column_names)

tokenized_valid.set_format("torch", columns=cols)
tokenized_test.set_format("torch", columns=cols)

valid_loader = DataLoader(tokenized_valid, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)
test_loader  = DataLoader(tokenized_test,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)


# functions

In [ ]:
s = np.array([np.random.choice(np.arange(num_of_client), size=10, replace=False) for _ in range(200)])
def get_trained_client(setting,numberOfClient,numberParticpate):

    clients = list(range(numberOfClient))

    if setting == 1:
        return clients

    elif setting == 2:
        return  random.sample(clients, numberParticpate)

    elif setting == 3:
        count = random.randint(1, len(clients))
        return random.sample(clients, count)
    elif setting == 4:
        senario = s
        return senario[numberParticpate]
    else:
        raise ValueError("no existing setting that can be used")

In [ ]:
@torch.no_grad()
def evaluate_valid_loss(model, loader: DataLoader, device):
    """
    Computes mean token-level negative log-likelihood (NLL)
    over the validation set, without using HuggingFace's
    built-in loss.

    Returns:
        valid_loss (float): mean NLL over all non-masked tokens
    """
    model.eval()

    total_nll = 0.0
    total_tokens = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        # logits: [batch_size, seq_len, vocab_size]
        logits = outputs.logits

        # Compute log-probabilities
        log_probs = F.log_softmax(logits, dim=-1)

        # Flatten tensors
        labels_flat = labels.view(-1)
        log_probs_flat = log_probs.view(-1, log_probs.size(-1))

        # Mask out ignored labels (-100)
        valid_mask = labels_flat != -100
        valid_labels = labels_flat[valid_mask]

        # Gather log-probabilities of the true labels
        token_log_probs = log_probs_flat[valid_mask, valid_labels]

        # Accumulate NLL
        total_nll += -token_log_probs.sum()
        total_tokens += valid_labels.numel()

    # Mean NLL over entire validation set
    valid_loss = (total_nll / total_tokens).item()

    return valid_loss

In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def evaluate_model(model, device, test_loader):
    """
    Hugging Face–compatible evaluation function

    Args:
        model: Hugging Face model (e.g. AutoModelForSequenceClassification)
        device: torch.device
        test_loader: DataLoader (HF-style batches)
    Returns:
        dict with validation_loss, accuracy, f1
    """
    model.to(device)
    model.eval()

    all_logits = []
    all_labels = []
    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for batch in test_loader:
            # HF batches are dicts
            inputs = {k: v.to(device) for k, v in batch.items()}

            # Forward pass (HF computes loss internally)
            outputs = model(**inputs)

            loss = outputs.loss
            logits = outputs.logits
            labels = inputs["labels"]

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            all_logits.append(logits.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    avg_loss = total_loss / total_samples

    all_logits = np.concatenate(all_logits, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    preds = np.argmax(all_logits, axis=1)

    metrics = {
        "validation_loss": avg_loss,
        "accuracy": accuracy.compute(
            predictions=preds,
            references=all_labels
        )["accuracy"],
        "f1": f1.compute(
            predictions=preds,
            references=all_labels
        )["f1"]
    }

    return metrics

In [ ]:
def _int_counts_from_simplex(prob, total, rng):
    """
    Convert simplex vector prob into integer counts that sum to total
    using floor + remainder distribution.
    """
    raw = prob * total
    counts = np.floor(raw).astype(int)
    rem = total - counts.sum()
    if rem > 0:
        frac = raw - counts
        # deterministic tie-breaking with rng noise
        order = np.argsort(-frac + 1e-12 * rng.random(len(frac)))
        counts[order[:rem]] += 1
    return counts

def dirichlet_partition_both(
    labels,
    num_clients: int = 10,
    alpha: float = 0.5,
    beta: float | None = None,
    prior: str = "uniform",      # you asked for uniform
    min_client_size: int = 1,
    seed: int = 42,
):

    rng = np.random.default_rng(seed)
    y = np.asarray(labels)
    N = len(y)

    # Map labels to 0..L-1
    classes = np.unique(y)
    L = len(classes)
    to_int = {c: i for i, c in enumerate(classes)}
    y_int = np.array([to_int[v] for v in y], dtype=int)

    # Prior p over labels (uniform per your requirement)
    if prior == "uniform":
        p = np.ones(L) / L
    elif prior == "empirical":
        cnt = np.bincount(y_int, minlength=L)
        p = cnt / cnt.sum()
    else:
        raise ValueError("prior must be 'uniform' or 'empirical'.")

    # ----- quantity sizes -----
    if beta is None:
        # balanced: same or very similar number of examples
        base = N // num_clients
        rem = N % num_clients
        sizes = [base + (1 if i < rem else 0) for i in range(num_clients)]
        z = None
    else:

        z = rng.dirichlet(np.ones(num_clients) * beta)
        sizes = _int_counts_from_simplex(z, N, rng).tolist()

        # enforce min_client_size if requested
        if min_client_size > 1:
            if min_client_size * num_clients > N:
                raise ValueError("min_client_size too large for total dataset size.")
            # raise small clients up to min by taking from larger ones
            deficit = sum(max(0, min_client_size - s) for s in sizes)
            if deficit > 0:
                order = np.argsort(-np.array(sizes))
                for i in order:
                    if deficit <= 0:
                        break
                    spare = sizes[i] - min_client_size
                    if spare > 0:
                        take = min(spare, deficit)
                        sizes[i] -= take
                        deficit -= take
                for j in range(num_clients):
                    need = max(0, min_client_size - sizes[j])
                    sizes[j] += need

            # re-fix sum exactly to N
            diff = N - sum(sizes)
            if diff > 0:
                for _ in range(diff):
                    sizes[rng.integers(num_clients)] += 1
            elif diff < 0:
                diff = -diff
                for _ in range(diff):
                    candidates = [i for i, s in enumerate(sizes) if s > min_client_size]
                    i = rng.choice(candidates)
                    sizes[i] -= 1

    # ----- label distributions -----

    conc = np.clip(alpha * p, 1e-12, None)  # ensure positive concentration
    q = rng.dirichlet(conc, size=num_clients)

    # Pools of indices per label (for sampling without replacement)
    pools = []
    for k in range(L):
        idxs = np.where(y_int == k)[0]
        rng.shuffle(idxs)
        pools.append(list(idxs))

    # Allocate without replacement according to q_j
    client_indices = []
    for j in range(num_clients):
        m = sizes[j]
        desired = rng.multinomial(m, q[j])

        assigned = []
        # take desired from each pool
        for k in range(L):
            take = min(desired[k], len(pools[k]))
            if take > 0:
                assigned.extend(pools[k][:take])
                pools[k] = pools[k][take:]

        while len(assigned) < m:
            avail = np.array([len(pools[k]) for k in range(L)])
            if avail.sum() == 0:
                break
            mask = (avail > 0).astype(float)
            probs = q[j] * mask
            probs = probs / probs.sum()
            k = rng.choice(L, p=probs)
            assigned.append(pools[k].pop())  # remove => without replacement

        rng.shuffle(assigned)
        client_indices.append(assigned)

    return client_indices, q, sizes, classes, z

In [ ]:

def make_client_shards(df, client_indices):
    """
    df: pandas.DataFrame
    client_indices: list[list[int]]  (row indices per client)
    returns: list[pandas.DataFrame]
    """
    return [df.iloc[idxs].reset_index(drop=True) for idxs in client_indices]


In [ ]:
client_idxs, q, sizes, classes, z = dirichlet_partition_both(
    labels=train_df["label"],
    num_clients=num_of_client,
    alpha=0.5,          # strong label non-IID
    beta=None,          # balanced client sizes
    prior="uniform",
    seed=seed
)
print("sizes:", sizes, "min/max:", min(sizes), max(sizes))

In [ ]:
all_ids = np.concatenate([np.array(x) for x in client_idxs])
print("Assigned:", len(all_ids), "Unique:", len(np.unique(all_ids)))

In [ ]:
client_shards = make_client_shards(train_df, client_idxs)

print("Client sizes:", [len(c) for c in client_shards])
print("Min/Max:", min(len(c) for c in client_shards), max(len(c) for c in client_shards))

print("Example client[0]:")
print(client_shards[0].iloc[0])


In [ ]:
train_proc = [ Dataset.from_pandas(train_ds, preserve_index=False) for train_ds in client_shards]
train_proc = [ train_ds.map(preprocess_fn, batched=True, remove_columns=train_ds.column_names) for train_ds in train_proc]
for i in train_proc:
        i.set_format(type="torch", columns=cols)

In [ ]:
train_loader = [DataLoader(i, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator) for i in train_proc]

# model realated function

In [ ]:
def base_model_loader(adapter_dir: str):
    model = AutoModelForSequenceClassification.from_pretrained(adapter_dir,num_labels=2)
    return model

In [ ]:
def _move_optimizer_state_to_device(opt_state, device):
    for state in opt_state.get("state", {}).values():
        for k, v in list(state.items()):
            if torch.is_tensor(v):
                state[k] = v.to(device, non_blocking=True)
            elif isinstance(v, (list, tuple)):
                state[k] = [t.to(device, non_blocking=True) if torch.is_tensor(t) else t for t in v]
            elif isinstance(v, dict):
                for kk, vv in list(v.items()):
                    if torch.is_tensor(vv):
                        v[kk] = vv.to(device, non_blocking=True)

In [ ]:
# ================= FEDADAMW STATE IO =================

FEDADAMW_STATE_FILE = "fedadamw_state.pt"

def get_fedadamw_state_path(global_adapter_dir):
    return os.path.join(global_adapter_dir, "fedadamw_state.pt")



In [ ]:
def save_full_model_state_dict(state_dict, out_dir: str):
    """
    Save FULL model weights (HF-compatible).
    Deletes existing directory to avoid stale files.
    """
    if os.path.exists(out_dir):
        print("removed")
        shutil.rmtree(out_dir)

    os.makedirs(out_dir, exist_ok=True)

    out_path = os.path.join(out_dir, "pytorch_model.bin")
    torch.save(state_dict, out_path)


def load_full_model_state_dict(model_dir: str, map_location="cpu"):
    """
    Load FULL model weights from HF-style directory.
    """
    bin_path = os.path.join(model_dir, "pytorch_model.bin")
    safe_path = os.path.join(model_dir, "model.safetensors")

    if os.path.exists(bin_path):
        return torch.load(bin_path, map_location=map_location)
    elif safe_load is not None and os.path.exists(safe_path):
        return safe_load(safe_path)
    else:
        raise FileNotFoundError(f"No model weights found in {model_dir}")


# ============================================================
# 2) FEDAVG OVER FULL MODELS (CPU SAFE)
# ============================================================

@torch.no_grad()
def average_full_model_states(model_dirs, weights=None):
    """
    FedAvg over FULL model parameters on CPU.
    Safely skips non-floating tensors.
    """
    n = len(model_dirs)
    if n == 0:
        raise ValueError("No model dirs provided.")

    if weights is None:
        weights = [1.0 / n] * n
    else:
        s = sum(weights)
        weights = [w / s for w in weights]

    avg = load_full_model_state_dict(model_dirs[0], map_location="cpu")

    for k, v in avg.items():
        if torch.is_floating_point(v):
            avg[k] = v.float() * weights[0]

    for w, d in zip(weights[1:], model_dirs[1:]):
        sd = load_full_model_state_dict(d, map_location="cpu")
        for k in avg:
            if torch.is_floating_point(avg[k]):
                avg[k].add_(sd[k].float(), alpha=w)

    return avg


# ============================================================
# 4) AGGREGATION CONTROLLER (FULL MODELS ONLY)
# ============================================================
def add_parameters(model, edge, ratio):
  for server_param, edge_param in zip(model.parameters(), edge.parameters()):
    server_param.data += edge_param.data.clone() * ratio

def sub_parameters(model, edge, ratio):
  for server_param, edge_param in zip(model.parameters(), edge.parameters()):
    server_param.data -= edge_param.data.clone() * ratio

def model_difference(model1,model2,n):
  sub_parameters(model1,model2,1)
  ed = f"diff_{str(n)}"
  model1.save_pretrained(ed)
  return ed

def aggregate_adapters(
    selected_model_dirs,
    algorithm: str,
    global_model_dir: str,
    train_loaders=None,
    client_ids=None,
    wagg_set=None,
    base_model_loader=None,
    device="cuda",
    training_losses=None,
):
    """
    Supported algorithms:
      - sophia
      - fedavg
      - wagg
      - fednolowe
      - fedamaw (FedAdamW)
    """

    assert base_model_loader is not None
    assert len(selected_model_dirs) > 0

    algorithm = algorithm.lower()

    # Load current global model (x_r)
    global_model = base_model_loader(global_model_dir)

    # ============================================================
    # SOPHIA / FedAvg (uniform)
    # ============================================================
    if algorithm in {"sophia", "fedavg"}:
        for p in global_model.parameters():
            if torch.is_floating_point(p):
                p.data.zero_()

        w = 1.0 / len(selected_model_dirs)

        for d in selected_model_dirs:
            client_model = base_model_loader(d)
            add_parameters(global_model, client_model, w)
            del client_model

        global_model.save_pretrained(global_model_dir)
        del global_model
        torch.cuda.empty_cache()
        return

    # ============================================================
    # WAGG
    # ============================================================
    if algorithm == "wagg":
        assert wagg_set is not None

        all_dirs = selected_model_dirs + [global_model_dir]
        scores = []

        for d in all_dirs:
            m = base_model_loader(d).to(device)
            loss = evaluate_valid_loss(m, wagg_set, device)
            scores.append(1.0 / (loss + 1e-12))
            del m
            torch.cuda.empty_cache()

        s = sum(scores)
        weights = [x / s for x in scores]

        for p in global_model.parameters():
            if torch.is_floating_point(p):
                p.data.zero_()

        for d, w in zip(all_dirs, weights):
            m = base_model_loader(d)
            add_parameters(global_model, m, w)
            del m

        global_model.save_pretrained(global_model_dir)
        del global_model
        torch.cuda.empty_cache()
        return

    # ============================================================
    # FEDNOLOWE
    # ============================================================
    if algorithm == "fednolowe":
        losses = np.array(training_losses, dtype=np.float64)

        if losses.sum() == 0:
            weights = np.ones(len(losses)) / len(losses)
        else:
            weights = 1.0 - losses / losses.sum()
            weights /= weights.sum()

        for p in global_model.parameters():
            if torch.is_floating_point(p):
                p.data.zero_()

        for d, w in zip(selected_model_dirs, weights):
            m = base_model_loader(d)
            add_parameters(global_model, m, w)
            del m

        global_model.save_pretrained(global_model_dir)
        del global_model
        torch.cuda.empty_cache()
        return

    # ============================================================
    # FEDADAMW (fedamaw)
    # ============================================================
    if algorithm in {"fedamaw", "fedadamw"}:

        # -------- save old global model x_r --------
        old_model = base_model_loader(global_model_dir)
        old_state = {
            k: v.clone()
            for k, v in old_model.state_dict().items()
            if torch.is_floating_point(v)
        }

        # -------- FedAvg to get x_{r+1} --------
        for p in global_model.parameters():
            if torch.is_floating_point(p):
                p.data.zero_()

        w = 1.0 / len(selected_model_dirs)

        for d in selected_model_dirs:
            m = base_model_loader(d)
            add_parameters(global_model, m, w)
            del m

        new_state = global_model.state_dict()

        # -------- compute ΔG = (x_{r+1} - x_r) / lr --------
        lr_used = None
        momen_m = {}

        for d in selected_model_dirs:
            state_path = os.path.join(d, "client_fedadamw_state.pt")
            if os.path.exists(state_path):
                lr_used = torch.load(state_path, map_location="cpu")["lr"]
                break

        if lr_used is None:
            raise RuntimeError("FedAdamW: cannot find client learning rate")

        for k in old_state:
            momen_m[k] = (new_state[k] - old_state[k]) / lr_used

        # -------- aggregate mean(v) --------
        momen_v = {}
        for d in selected_model_dirs:
            state = torch.load(
                os.path.join(d, "client_fedadamw_state.pt"),
                map_location="cpu",
            )["client_v"]

            for k, v in state.items():
                momen_v[k] = momen_v.get(k, 0.0) + v / len(selected_model_dirs)

        # -------- save everything --------
        global_model.save_pretrained(global_model_dir)
        save_fedadamw_state(global_model_dir, momen_m, momen_v)

        del global_model
        del old_model
        torch.cuda.empty_cache()
        return

    # ============================================================
    raise ValueError(f"Unknown algorithm: {algorithm}")

In [ ]:
@torch.no_grad()
def aggregate_fedadamw_symbolic(
    selected_model_dirs,
    global_model_dir,
    base_model_loader,
    lr,        # η
    K,         # local steps
):
    S = len(selected_model_dirs)

    # ---- load x^r ----
    global_model = base_model_loader(global_model_dir)
    x_r = {
        k: v.clone()
        for k, v in global_model.state_dict().items()
        if torch.is_floating_point(v)
    }

    # ---- compute x^{r+1} (Eq. 2) ----
    for p in global_model.parameters():
        if torch.is_floating_point(p):
            p.data.zero_()

    for d in selected_model_dirs:
        client_model = base_model_loader(d)
        for p_s, p_c in zip(global_model.parameters(), client_model.parameters()):
            if torch.is_floating_point(p_s):
                p_s.data.add_(p_c.data, alpha=1.0 / S)
        del client_model

    x_r_plus_1 = global_model.state_dict()

    # ---- compute ΔG^r (Eq. 1) ----
    delta_G = {}
    for k in x_r:
        delta_G[k] = -(x_r_plus_1[k] - x_r[k]) / (S * K * lr)

    # ---- aggregate v (Eq. 3) ----
    v_bar = {}
    for d in selected_model_dirs:
        state = torch.load(
            os.path.join(d, "client_fedadamw_state.pt"),
            map_location="cpu"
        )["client_v"]

        for k, v in state.items():
            v_bar[k] = v_bar.get(k, 0.0) + v / S

    # ---- save ----
    global_model.save_pretrained(global_model_dir)
    torch.save(
        {
            "delta_G": delta_G,
            "v_bar": v_bar,
        },
        os.path.join(global_model_dir, "fedadamw_state.pt"),
    )

In [ ]:
def load_fedadamw_state(global_adapter_dir, model, lr, K):
    """
    Paper-symbol loader.

    Returns:
        delta_G : Δ_G^r
        v_bar   : \bar v^r
    """
    path = os.path.join(global_adapter_dir, "fedadamw_state.pt")

    if not os.path.exists(path):
        # Round 0 initialization (Δ_G^0 = 0, v̄^0 = 0)
        delta_G = {
            k: torch.zeros_like(v)
            for k, v in model.state_dict().items()
            if torch.is_floating_point(v)
        }

        v_bar = {
            k: torch.tensor(0.0)
            for k, v in model.state_dict().items()
            if torch.is_floating_point(v)
        }

        return delta_G, v_bar

    state = torch.load(path, map_location="cpu")

    return state["delta_G"], state["v_bar"]

In [ ]:
def get_client_state_paths(edge: int, base_state_dir="client_state"):
    state_dir = os.path.join(base_state_dir, f"edge_{edge}")
    os.makedirs(state_dir, exist_ok=True)
    opt_path = os.path.join(state_dir, "optimizer.pt")
    return state_dir, opt_path

def get_round_adapter_dir(round_id: int, edge: int, cache_root="round_cache"):
    d = os.path.join(cache_root, f"round_{round_id}", f"edge_{edge}")
    os.makedirs(d, exist_ok=True)
    return d

# train loop

In [ ]:
print("main")
test_acc = evaluate_model(model, device,test_loader)
print(f" global_loss={test_acc["validation_loss"]:.4f} global_acc = {test_acc["accuracy"]:.4f} f1={test_acc["f1"]:.4f}")


In [ ]:
def train_secondorder_fedsophia(
    base_model_loader,          # function(adapter_dir) -> model
    global_adapter_dir: str,    # current global adapter checkpoint directory
    train_loaders,
    test_loader,
    valid_loaders=None,
    algorithm="FedAdamW",
    grad_clip=1.0,
    lr=3e-5,
    epochs=1,
    rounds=10,
    num_clients=10,
    clients_per_round=4,
    seed=42,
    device="cuda",
    k=10,
):
    """
    FedAdamW-style local training loop.

    Assumes you already have:
      - load_fedadamw_state(global_adapter_dir, model)
      - get_trained_client(...)
      - get_round_adapter_dir(...)
      - aggregate_adapters(...)
      - evaluate_model(...)
      - cosine_lr(...)
    """

    if device == "cuda" and not torch.cuda.is_available():
        device = "cpu"

    results = []

    for globalRound in range(rounds):
        print(f"\n##################  Round {globalRound}  ########################")

        chosen = get_trained_client(4, num_clients, globalRound)

        training_losses = []
        selected_adapter_dirs = []
        selected_client_ids = []

        # Round learning rate
        new_lr = cosine_lr(
            initial_lr=lr,
            round_idx=globalRound,
            total_rounds=rounds,
            eta_min=1e-8,
        )

        for edge in range(num_clients):
            if edge not in chosen:
                continue

            print(f"\n################## edge {edge} ########################")

            out_adapter_dir = get_round_adapter_dir(globalRound, edge)

            # ---- load model ----
            model = base_model_loader(adapter_dir=global_adapter_dir).to(device)
            model.train()

            # ---- load FedAdamW server state ----
            # Round 0: should initialize zeros.
            # Later rounds: should load momen_m and momen_v saved by aggregation.

            delta_G, momen_v = load_fedadamw_state(
                global_adapter_dir,
                model,
                lr=new_lr,
                K=epochs * len(train_loaders[edge]),
            )


            for k in delta_G:
                delta_G[k] = delta_G[k].to(device)

            params = [p for p in model.parameters() if p.requires_grad]

            if len(params) == 0:
                raise ValueError(
                    "No trainable parameters found. "
                    "Check requires_grad settings or LoRA/adapters."
                )

            optimizer = torch.optim.AdamW(
                params,
                lr=new_lr,
                weight_decay=5e-6,
                amsgrad=False,
            )

            # ============================================================
            # Initialize AdamW state for FedAdamW
            #   m = 0
            #   v = server aggregated momen_v
            # ============================================================
            param_id_to_name = {
                id(p): n
                for n, p in model.named_parameters()
                if p.requires_grad
            }

            for group in optimizer.param_groups:
                for p in group["params"]:
                    if id(p) not in param_id_to_name:
                        continue

                    param_name = param_id_to_name[id(p)]
                    state = optimizer.state[p]

                    # Required by PyTorch AdamW
                    state["step"] = torch.zeros(
                        (),
                        dtype=torch.float32,
                        device=p.device,
                    )

                    # FedAdamW: reset first moment every round
                    state["exp_avg"] = torch.zeros_like(
                        p.data,
                        memory_format=torch.preserve_format,
                    )

                    # FedAdamW: initialize second moment from server momen_v
                    if param_name in momen_v:
                        v_value = momen_v[param_name]

                        if torch.is_tensor(v_value):
                            v_value = v_value.detach().cpu()

                            # Case 1: scalar mean(v)
                            if v_value.numel() == 1:
                                state["exp_avg_sq"] = torch.full_like(
                                    p.data,
                                    float(v_value.item()),
                                    memory_format=torch.preserve_format,
                                )

                            # Case 2: full tensor v
                            elif tuple(v_value.shape) == tuple(p.data.shape):
                                state["exp_avg_sq"] = v_value.to(
                                    device=p.device,
                                    dtype=p.dtype,
                                ).clone()

                            # Shape mismatch fallback
                            else:
                                print(
                                    f"[Warning] momen_v shape mismatch for {param_name}. "
                                    f"Expected {tuple(p.data.shape)}, got {tuple(v_value.shape)}. "
                                    "Using zeros."
                                )
                                state["exp_avg_sq"] = torch.zeros_like(
                                    p.data,
                                    memory_format=torch.preserve_format,
                                )

                        else:
                            # Python float/int scalar
                            state["exp_avg_sq"] = torch.full_like(
                                p.data,
                                float(v_value),
                                memory_format=torch.preserve_format,
                            )

                    else:
                        # Round 0 or missing key
                        state["exp_avg_sq"] = torch.zeros_like(
                            p.data,
                            memory_format=torch.preserve_format,
                        )

            gamma = 0.6

            # Number of local optimizer steps used for correction scaling
            local_steps = max(1, epochs * len(train_loaders[edge]))
            global_steps = 0

            # ============================================================
            # Local training
            # ============================================================
            for epoch in range(1, epochs + 1):
                running_loss = 0.0

                for step, batch in enumerate(train_loaders[edge], start=1):
                    input_ids = batch["input_ids"].to(device, non_blocking=True)
                    attention_mask = batch["attention_mask"].to(device, non_blocking=True)
                    labels = batch["labels"].to(device, non_blocking=True)

                    optimizer.zero_grad(set_to_none=True)

                    outputs = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels,
                    )

                    loss = outputs.loss
                    loss.backward()

                    if grad_clip is not None and grad_clip > 0:
                        torch.nn.utils.clip_grad_norm_(
                            model.parameters(),
                            max_norm=grad_clip,
                        )

                    # AdamW part:
                    # x <- x - lr * AdamW_direction
                    optimizer.step()

                    with torch.no_grad():
                        for n, p in model.named_parameters():
                            if not p.requires_grad:
                                continue
                            if n not in delta_G:
                                continue

                            p.data.add_(
                                delta_G[n].to(device=p.device, dtype=p.dtype),
                                alpha=-new_lr * gamma
                            )

                    running_loss += float(loss.detach().cpu())
                    global_steps += 1

                avg_train_loss = running_loss / max(1, len(train_loaders[edge]))

                val_acc = evaluate_model(model, device, test_loader)

                print(
                    f"Epoch {epoch}: "
                    f"train_loss={avg_train_loss:.4f} "
                    f"val_loss={val_acc['validation_loss']:.4f} "
                    f"acc={val_acc['accuracy']:.4f} "
                    f"f1={val_acc['f1']:.4f}"
                )

                training_losses.append(avg_train_loss)

            # ============================================================
            # Save client adapter/model
            # ============================================================
            model.save_pretrained(out_adapter_dir)

            # ============================================================
            # Save client mean(v) for server aggregation
            # ============================================================
            param_id_to_name = {
                id(param): name
                for name, param in model.named_parameters()
                if param.requires_grad
            }

            client_v = {}

            for p, state in optimizer.state.items():
                if "exp_avg_sq" not in state:
                    continue

                param_name = param_id_to_name.get(id(p), None)

                if param_name is None:
                    continue

                # Communication-efficient FedAdamW:
                # save scalar mean(v) per trainable tensor
                client_v[param_name] = (
                    state["exp_avg_sq"]
                    .detach()
                    .float()
                    .mean()
                    .cpu()
                )

            torch.save(
                {
                    "client_v": client_v,
                    "local_steps": global_steps,
                    "lr": new_lr,
                },
                os.path.join(out_adapter_dir, "client_fedadamw_state.pt"),
            )

            selected_adapter_dirs.append(out_adapter_dir)
            selected_client_ids.append(edge)

            # ---- free GPU memory ----
            del model
            del optimizer

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            gc.collect()

        # ============================================================
        # Server aggregation
        # ============================================================
        if len(selected_adapter_dirs) == 0:
            print(f"[Warning] No clients selected in round {globalRound}. Skipping aggregation.")
            continue



        aggregate_fedadamw_symbolic(
            selected_adapter_dirs,
            global_adapter_dir,base_model_loader,new_lr,len(train_loaders[edge]))

        # ============================================================
        # Evaluate global model
        # ============================================================
        global_model = base_model_loader(adapter_dir=global_adapter_dir).to(device)

        test_acc = evaluate_model(global_model, device, test_loader)
        results.append(test_acc)

        print(
            f"Round {globalRound}: "
            f"global_loss={test_acc['validation_loss']:.4f} "
            f"global_acc={test_acc['accuracy']:.4f} "
            f"f1={test_acc['f1']:.4f}")

        del global_model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        gc.collect()

        # ---- delete per-round cache to save disk ----
        round_dir = os.path.join("round_cache", f"round_{globalRound}")

        if os.path.exists(round_dir) and globalRound != rounds - 1:
            shutil.rmtree(round_dir, ignore_errors=True)

    return results

# half wagg

In [ ]:
#del model
torch.cuda.empty_cache()
gc.collect()
start_time = time.time()  # Record start time
results = train_secondorder_fedsophia(
    base_model_loader,          # function(adapter_dir) -> model (base + adapter loaded)
    global_adapter_dir = Main_path,    # current global adapter checkpoint directory
    train_loaders = train_loader,
    test_loader = test_loader,
    valid_loaders= valid_loader ,         # if you use it elsewhere
    algorithm="fedamaw",
    grad_clip=10,
    lr=LR,
    epochs=EPOCHS,
    rounds=rounds,
    num_clients=num_of_client,
    clients_per_round=num_of_Part,
    seed=seed,
    device=device)         # Call the function
end_time = time.time()  # Record end time

execution_time = end_time - start_time
print(f"Execution time: {execution_time:.6f} seconds")

In [ ]:
for i in range(len(results)):
    print(f"round {i} = {results[i]}")